<a href="https://colab.research.google.com/github/abhaysachan007/Fault-Detection-Challenge/blob/main/Fault_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install catboost xgboost lightgbm openpyxl

In [ ]:
# Fault Detection System - Stacked Ensemble Implementation
# --------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
import os
from sklearn.preprocessing import RobustScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
TRAIN_PATH = 'TRAIN.csv'
TEST_PATH = 'TEST.csv'
SUBMISSION_PATH = 'FINAL_SUBMISSION.csv'
RANDOM_STATE = 42

def load_data(path):
    """Robust data loader handling encoding issues."""
    if not os.path.exists(path):
        # Check for Excel alternative if CSV missing
        xlsx_path = path.replace('.csv', '.xlsx')
        if os.path.exists(xlsx_path):
             return pd.read_excel(xlsx_path)
        raise FileNotFoundError(f"File not found: {path}")

    try:
        return pd.read_csv(path)
    except Exception:
        # Fallback for encoding issues
        return pd.read_csv(path, encoding='latin1', on_bad_lines='skip')

def main():
    print("Initializing training pipeline...")

    # 1. Load Data
    train_df = load_data(TRAIN_PATH)
    test_df = load_data(TEST_PATH)

    # 2. Data Cleaning
    # Remove rows with missing targets and ensure integer type
    train_df = train_df.dropna(subset=['Class'])
    train_df['Class'] = train_df['Class'].astype(int)

    # 3. Feature Engineering
    features = [col for col in test_df.columns if col != 'ID']

    # Log-transform highly skewed features to handle outliers
    all_data = pd.concat([train_df[features], test_df[features]])
    skewness = all_data.apply(lambda x: x.skew()).sort_values(ascending=False)
    high_skew_cols = skewness[skewness > 10].index.tolist()

    if high_skew_cols:
        for col in high_skew_cols:
            train_df[col] = np.log1p(train_df[col])
            test_df[col] = np.log1p(test_df[col])

    # Prepare datasets
    X = train_df[features]
    y = train_df['Class']
    X_test = test_df[features]
    test_ids = test_df['ID']

    # 4. Scaling
    scaler = RobustScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features)

    # 5. Model Training (Ensemble)
    print("Training XGBoost...")
    xgb_clf = xgb.XGBClassifier(
        n_estimators=1200, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        n_jobs=-1, tree_method='hist'
    )
    xgb_clf.fit(X_scaled, y)
    preds_xgb = xgb_clf.predict_proba(X_test_scaled)[:, 1]

    print("Training LightGBM...")
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=1200, learning_rate=0.05, num_leaves=35,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
    lgb_clf.fit(X_scaled, y)
    preds_lgb = lgb_clf.predict_proba(X_test_scaled)[:, 1]

    print("Training CatBoost...")
    cat_clf = CatBoostClassifier(
        iterations=1200, learning_rate=0.05, depth=7,
        loss_function='Logloss', verbose=0, random_seed=RANDOM_STATE
    )
    cat_clf.fit(X_scaled, y)
    preds_cat = cat_clf.predict_proba(X_test_scaled)[:, 1]

    # 6. Ensemble Voting (Weighted Average)
    final_probs = (0.35 * preds_cat) + (0.35 * preds_xgb) + (0.30 * preds_lgb)
    final_predictions = (final_probs > 0.5).astype(int)

    # 7. Export Submission
    submission = pd.DataFrame({'ID': test_ids, 'CLASS': final_predictions})
    submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n✅ SUCCESS! Ensemble complete.")
    print(f"   Output saved to: {SUBMISSION_PATH}")

if __name__ == "__main__":
    main()

Initializing training pipeline...
Training XGBoost...
Training LightGBM...
Training CatBoost...

✅ SUCCESS! Ensemble complete.
   Output saved to: FINAL_SUBMISSION.csv
